# Exploring Attention and Contextual Embeddings for Social Media Sentiment Analysis

## Problem Statement
Social media platforms generate massive volumes of user-generated text expressing opinions and emotions. This assignment explores how attention mechanisms combined with contextual embeddings improve sentiment analysis on social media data.

## Objectives
1. Understand contextual embeddings in sentiment analysis
2. Analyze the role of attention mechanisms
3. Implement sentiment classifiers with and without attention
4. Compare model performance and interpretability

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Task 1: Data Preprocessing (1 mark)

In [2]:
# Load and preprocess data
columns = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding='latin-1', names=columns)

def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    return ' '.join(text.split()).lower().strip()

df['clean_text'] = df['text'].apply(clean_text)
df['sentiment'] = df['target'].map({0: 0, 4: 1})
df = df[df['clean_text'].str.len() > 0]

# Use smaller sample for faster training
df_sample = df.sample(n=2000, random_state=42).reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(
    df_sample['clean_text'], df_sample['sentiment'], 
    test_size=0.2, random_state=42, stratify=df_sample['sentiment']
)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Sample text: {X_train.iloc[0]}")

Training samples: 1600, Test samples: 400
Sample text: inshalla


In [3]:
# Dataset and DataLoader
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]
        
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

train_dataset = SentimentDataset(X_train, y_train, tokenizer)
test_dataset = SentimentDataset(X_test, y_test, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Task 2: Baseline Model (2 marks)

In [4]:
class BaselineModel(nn.Module):
    def __init__(self, n_classes=2):
        super(BaselineModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        # Freeze BERT for faster training
        for param in self.bert.parameters():
            param.requires_grad = False
        self.classifier = nn.Linear(768, n_classes)
    
    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(outputs.pooler_output)

baseline_model = BaselineModel().to(device)
print(f"Trainable parameters: {sum(p.numel() for p in baseline_model.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 1,538


In [ ]:
def train_model(model, train_loader, test_loader, epochs=1):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = correct / total
        
        # Evaluation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(input_ids, attention_mask)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_acc = correct / total
        print(f'Epoch {epoch+1}: Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')
    
    return train_acc, test_acc

print("Training Baseline Model...")
baseline_train_acc, baseline_test_acc = train_model(baseline_model, train_loader, test_loader)

Training Baseline Model...


## Task 3: Attention-Based Models (3.5 marks)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, hidden_size):
        super(SelfAttention, self).__init__()
        self.hidden_size = hidden_size
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.hidden_size ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_values = torch.matmul(attention_weights, V)
        
        return attended_values, attention_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads=4):
        super(MultiHeadAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        
        Q = self.query(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_values = torch.matmul(attention_weights, V)
        
        attended_values = attended_values.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.hidden_size
        )
        
        return self.out(attended_values), attention_weights.mean(dim=1)

class AttentionModel(nn.Module):
    def __init__(self, attention_type='self', n_classes=2):
        super(AttentionModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        for param in self.bert.parameters():
            param.requires_grad = False
        
        if attention_type == 'self':
            self.attention = SelfAttention(768)
        else:
            self.attention = MultiHeadAttention(768)
        
        self.classifier = nn.Linear(768, n_classes)
    
    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        sequence_output = outputs.last_hidden_state
        attended_output, attention_weights = self.attention(sequence_output)
        pooled_output = torch.mean(attended_output, dim=1)
        
        return self.classifier(pooled_output), attention_weights

# Initialize models
self_attention_model = AttentionModel(attention_type='self').to(device)
multi_head_model = AttentionModel(attention_type='multi_head').to(device)

print("Models initialized successfully!")

In [ ]:
def train_attention_model(model, train_loader, test_loader, epochs=1):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = correct / total
        
        # Evaluation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                outputs, _ = model(input_ids, attention_mask)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_acc = correct / total
        print(f'Epoch {epoch+1}: Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')
    
    return train_acc, test_acc

print("Training Self-Attention Model...")
self_train_acc, self_test_acc = train_attention_model(self_attention_model, train_loader, test_loader)

print("\nTraining Multi-Head Attention Model...")
multi_train_acc, multi_test_acc = train_attention_model(multi_head_model, train_loader, test_loader)

## Attention Visualization

In [ ]:
def visualize_attention(model, text, tokenizer):
    model.eval()
    encoding = tokenizer(text, truncation=True, padding='max_length', 
                        max_length=64, return_tensors='pt')
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs, attention_weights = model(input_ids, attention_mask)
        prediction = torch.softmax(outputs, dim=1)
    
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    attention = attention_weights[0].cpu().numpy()
    
    # Get valid tokens and attention weights
    valid_tokens = []
    valid_attention = []
    
    for i, token in enumerate(tokens):
        if token not in ['[PAD]', '[CLS]', '[SEP]'] and attention_mask[0][i] == 1:
            valid_tokens.append(token)
            valid_attention.append(attention[i].mean())
    
    return valid_tokens, valid_attention, prediction.cpu().numpy()[0]

# Example visualization
sample_text = "I thought the phone would be great, but the battery life is terrible"
print(f"Sample text: {sample_text}")

tokens, attention_weights, prediction = visualize_attention(self_attention_model, sample_text, tokenizer)

print(f"\nPrediction: {'Positive' if prediction[1] > prediction[0] else 'Negative'}")
print(f"Confidence: {max(prediction):.4f}")

# Plot attention weights
plt.figure(figsize=(12, 6))
plt.bar(range(len(tokens)), attention_weights)
plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
plt.title('Self-Attention Weights')
plt.ylabel('Attention Weight')
plt.tight_layout()
plt.show()

# Show top attended words
token_attention_pairs = list(zip(tokens, attention_weights))
token_attention_pairs.sort(key=lambda x: x[1], reverse=True)
print(f"\nTop 5 attended words:")
for token, weight in token_attention_pairs[:5]:
    print(f"  {token}: {weight:.4f}")

## Task 4: Comparative Analysis (3.5 marks)

In [ ]:
def evaluate_model(model, test_loader, is_attention_model=False):
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            if is_attention_model:
                outputs, _ = model(input_ids, attention_mask)
            else:
                outputs = model(input_ids, attention_mask)
            
            _, predicted = torch.max(outputs.data, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_predictions), np.array(all_labels)

# Evaluate all models
baseline_preds, baseline_labels = evaluate_model(baseline_model, test_loader, False)
self_preds, self_labels = evaluate_model(self_attention_model, test_loader, True)
multi_preds, multi_labels = evaluate_model(multi_head_model, test_loader, True)

# Calculate metrics
def calculate_metrics(predictions, labels):
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    return accuracy, precision, recall, f1

baseline_metrics = calculate_metrics(baseline_preds, baseline_labels)
self_metrics = calculate_metrics(self_preds, self_labels)
multi_metrics = calculate_metrics(multi_preds, multi_labels)

# Results comparison
results_df = pd.DataFrame({
    'Model': ['Baseline (BERT)', 'Self-Attention', 'Multi-Head Attention'],
    'Accuracy': [baseline_metrics[0], self_metrics[0], multi_metrics[0]],
    'Precision': [baseline_metrics[1], self_metrics[1], multi_metrics[1]],
    'Recall': [baseline_metrics[2], self_metrics[2], multi_metrics[2]],
    'F1-Score': [baseline_metrics[3], self_metrics[3], multi_metrics[3]]
})

print("Model Comparison Results:")
print(results_df.round(4))

In [ ]:
# Visualization of results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

axes[0, 0].bar(x - width, [baseline_metrics[0], baseline_metrics[1], baseline_metrics[2], baseline_metrics[3]], 
               width, label='Baseline', alpha=0.8)
axes[0, 0].bar(x, [self_metrics[0], self_metrics[1], self_metrics[2], self_metrics[3]], 
               width, label='Self-Attention', alpha=0.8)
axes[0, 0].bar(x + width, [multi_metrics[0], multi_metrics[1], multi_metrics[2], multi_metrics[3]], 
               width, label='Multi-Head', alpha=0.8)

axes[0, 0].set_title('Model Performance Comparison')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Confusion Matrix for Multi-Head model
cm = confusion_matrix(multi_labels, multi_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1])
axes[0, 1].set_title('Confusion Matrix - Multi-Head Attention')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Actual')

# 3. Sample predictions comparison
test_examples = [
    "This movie is absolutely amazing!",
    "Worst experience ever!",
    "The service was okay",
    "Great product, highly recommend!"
]

model_names = ['Baseline', 'Self-Attention', 'Multi-Head']
models = [baseline_model, self_attention_model, multi_head_model]
predictions_matrix = []

for text in test_examples:
    text_preds = []
    for i, model in enumerate(models):
        model.eval()
        encoding = tokenizer(text, truncation=True, padding='max_length', 
                           max_length=64, return_tensors='pt')
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        
        with torch.no_grad():
            if i == 0:  # Baseline
                outputs = model(input_ids, attention_mask)
            else:  # Attention models
                outputs, _ = model(input_ids, attention_mask)
            pred = torch.softmax(outputs, dim=1)[0][1].item()  # Positive probability
        text_preds.append(pred)
    predictions_matrix.append(text_preds)

predictions_matrix = np.array(predictions_matrix)
im = axes[1, 0].imshow(predictions_matrix, cmap='RdYlBu_r', aspect='auto')
axes[1, 0].set_title('Model Predictions (Positive Probability)')
axes[1, 0].set_xticks(range(len(model_names)))
axes[1, 0].set_xticklabels(model_names)
axes[1, 0].set_yticks(range(len(test_examples)))
axes[1, 0].set_yticklabels([ex[:20] + '...' for ex in test_examples])
plt.colorbar(im, ax=axes[1, 0])

# 4. Performance summary
axes[1, 1].axis('off')
summary_text = f"""
Performance Summary:

Baseline Model:
• Accuracy: {baseline_metrics[0]:.3f}
• F1-Score: {baseline_metrics[3]:.3f}

Self-Attention Model:
• Accuracy: {self_metrics[0]:.3f}
• F1-Score: {self_metrics[3]:.3f}

Multi-Head Attention:
• Accuracy: {multi_metrics[0]:.3f}
• F1-Score: {multi_metrics[3]:.3f}

Key Findings:
• Attention mechanisms provide interpretability
• Multi-head attention captures diverse patterns
• BERT contextual embeddings are effective
"""
axes[1, 1].text(0.1, 0.9, summary_text, transform=axes[1, 1].transAxes, 
                fontsize=10, verticalalignment='top', fontfamily='monospace')

plt.tight_layout()
plt.show()

## Key Findings and Conclusions

### Performance Analysis:
1. **Contextual Embeddings**: BERT effectively captures context-dependent meanings
2. **Attention Mechanisms**: Provide interpretability by highlighting important words
3. **Model Comparison**: All models show competitive performance on sentiment analysis

### Attention Benefits:
1. **Interpretability**: Attention weights reveal sentiment-bearing words
2. **Focus**: Models learn to focus on key sentiment indicators
3. **Context Understanding**: Better handling of negations and sentiment shifts

### Social Media Challenges Addressed:
1. **Informal Language**: Contextual embeddings handle abbreviations
2. **Context Dependency**: Attention resolves ambiguous expressions
3. **Noise Filtering**: Attention focuses on relevant sentiment signals

### Recommendations:
1. Use attention mechanisms for interpretable sentiment analysis
2. Multi-head attention provides diverse perspective capture
3. Combine with domain-specific fine-tuning for better performance